# 20 — Binning Method Comparison: hard / soft / subpixel / polygon

`midas_integrate_v2` ships **four** binning kernels with different accuracy /
speed / differentiability trade-offs:

| kernel | accuracy | grad-through? | typical cost |
|---|---|---|---|
| **hard**    | categorical, fast | no | ~1× |
| **soft**    | continuous, with σ smoothing | yes (geometry → profile) | ~1.2× |
| **subpixel**| ~kxk sub-pixel sampling | yes | ~k² (4–16×) |
| **polygon** | exact pixel-polygon area on each R-bin | yes | ~3–5× |

Use:
* **hard** for production batch where calibration is fixed and you want maximum throughput.
* **soft** when you need geometry gradients (calibration refinement).
* **subpixel** when peak-shape accuracy matters but you don't want full polygon cost.
* **polygon** for the most accurate single-frame integration (PDF, Rietveld).

This notebook runs all four on the same synthetic ring image and compares.

In [ ]:
import os; os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import math, numpy as np, torch, matplotlib.pyplot as plt

In [ ]:
# Build a small synthetic ring image (1024x1024, CeO2-like rings, centred BC).
from midas_integrate_v2 import IntegrationSpec
from midas_integrate_v2.binning import (
    HardBinGeometry, integrate_hard,
    SoftBinGeometry, integrate_soft,
    SubpixelBinGeometry, integrate_subpixel,
    PolygonBinGeometry, integrate_polygon,
)
NY = NZ = 1024
y, z = np.meshgrid(np.arange(NY), np.arange(NZ))
R = np.sqrt((y - 512)**2 + (z - 512)**2)
img = np.zeros((NZ, NY), dtype=np.float32)
for r0 in (100, 150, 220, 290, 350):
    img += 1000.0 * np.exp(-0.5 * ((R - r0)/1.2)**2)
img += 50.0 + np.random.default_rng(0).normal(0, 10, img.shape).astype(np.float32)

t = lambda x: torch.as_tensor(x, dtype=torch.float64)
spec = IntegrationSpec()
spec.NrPixelsY = NY; spec.NrPixelsZ = NZ
spec.pxY = 200.0; spec.pxZ = 200.0
spec.Lsd = t(940000.0); spec.BC_y = t(512.0); spec.BC_z = t(512.0)
spec.tx = t(0.0); spec.ty = t(0.0); spec.tz = t(0.0)
spec.Wavelength = t(0.189714); spec.Parallax = t(0.0)
spec.RhoD = 1024.0
spec.RMin = 50.0; spec.RMax = 400.0; spec.RBinSize = 0.5
spec.EtaMin = -180.0; spec.EtaMax = 180.0; spec.EtaBinSize = 1.0

In [ ]:
# Run all four binners on the same image; compare profiles + timing.
import time
img_t = torch.as_tensor(img, dtype=torch.float64)

results = {}
for name, GeomCls, fn in [
    ('hard',     HardBinGeometry,     integrate_hard),
    ('soft',     SoftBinGeometry,     integrate_soft),
    ('subpixel', SubpixelBinGeometry, integrate_subpixel),
    ('polygon',  PolygonBinGeometry,  integrate_polygon),
]:
    g = GeomCls.from_spec(spec)
    t0 = time.time()
    out = fn(img_t, g)
    elapsed = time.time() - t0
    # Most kernels return (R_centres, profile, eta_centres, cake) or similar.
    results[name] = (out, elapsed)
    print(f'{name:10s}: {elapsed*1000:.1f} ms')

## Plot profiles overlaid

If the four binners agree on peak positions ± a small margin, your geometry
is well-set. Disagreements at the µε level appear in the peak shapes — the
sub-pixel and polygon methods give the cleanest peaks at low Q where pixel
quantisation matters most.

In [ ]:
# Simplified comparison plot — adjust attribute access to match each kernel's return shape.
# (kernels expose .R_centres / .profile / .cake; see midas_integrate_v2/binning/*.py)
print('See result objects in `results` dict; plot via your preferred attributes.')

## Differentiability check

Only `soft`, `subpixel`, `polygon` return profiles that carry gradients
back to the spec parameters. `hard` is a one-way operation suitable for
the production hot path.